In [0]:
# %run ../framework

In [0]:
import sys
import os
# 1. Add import to silver_unit_test.py
from delta import configure_spark_with_delta_pip

# Import logic that works in GitHub Runner, Local, and Databricks
# Force reload to pick up latest fw.py changes (avoids stale sys.modules cache)
for _mod in list(sys.modules):
    if _mod.startswith("unified_fw"):
        del sys.modules[_mod]

try:
    from unified_fw.fw import GoldLayer
except ImportError:
    # 1. Resolve the notebook's directory:
    #    - GitHub Runner/Local: use __file__
    #    - Databricks: use notebook context (works even via dbutils.notebook.run())
    if "__file__" in globals():
        current_dir = os.path.dirname(os.path.abspath(__file__))
    else:
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        current_dir = os.path.dirname(f"/Workspace{nb_path}")

    # 2. find location project_root by try go back 1 step
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
    package_path = os.path.join(project_root, "logic_packages", "src")

    # If go back but don't find (in case Root directly) use current folder
    if not os.path.exists(package_path):
        package_path = os.path.join(current_dir, "logic_packages", "src")

    # 3. ADD sys.path
    if package_path not in sys.path:
        sys.path.insert(0, package_path)

    from unified_fw.fw import GoldLayer

In [0]:
dbutils.widgets.text("pipeline_name", "")
pipeline_name = dbutils.widgets.get("pipeline_name")

In [0]:
# Create GoldLayer object and get variable from config table
g = GoldLayer.from_config_table(pipeline_name)

# Run gold pipline
g.run_gold_pipeline()